# Step 16: Intermediate: Tool Reasoning and Schema Design

        _Instructor Solution Notebook_  
        Level: **Intermediate**  
        Estimated time: **90 minutes**

        ## Learning Objectives
        - [ ] Design tools with schema clarity and predictable outputs.
- [ ] Compare loose text returns with structured return contracts.
- [ ] Reason about tool ambiguity, overlap, and sequencing.
- [ ] Practice multi-step tool plans that the model can execute safely.

        ## Prerequisites
        - Core Steps 4-7 completed.
- Comfort reading typed function signatures.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Reflection & Next Experiments
8. Summary & Key Takeaways
9. Additional Resources


## Setup & Imports

Supplemental notebooks assume the core helpers are available and that you have already worked
through the matching core lessons. The import cell intentionally keeps the same bootstrap shape
as the core course so you can focus on the deeper pattern rather than environment setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

This notebook goes beyond basic tool registration and focuses on how tool contract design changes model behavior. The goal is to make tools easier to choose, easier to compose, and easier to debug.

This notebook is intentionally deeper than the core path. Expect more design discussion,
more implementation sections, and more open-ended exercises.


## Part 2: Core Implementation

### Compare weak and strong tool contracts


In [ ]:
from agent_framework import tool
from pydantic import BaseModel

@tool
def weak_search_docs(query: str) -> str:
    """Search docs and return a free-form sentence."""
    return f"Found some notes about {query}, maybe in the docs folder."

class SearchResult(BaseModel):
    source: str
    relevance: float
    summary: str

@tool
def strong_search_docs(query: str) -> dict:
    """Search local course docs and return a stable structured payload."""
    return SearchResult(
        source="docs/references.md",
        relevance=0.82,
        summary=f"Structured notes relevant to {query}",
    ).model_dump()

print_json(
    {
        "weak_return": weak_search_docs("tool design"),
        "strong_return": strong_search_docs("tool design"),
    },
    title="Contract comparison",
)


## Part 2: Core Implementation

### Design a tool suite with minimal overlap


In [ ]:
@tool
def list_course_docs() -> list[str]:
    """List high-level course docs available in the repo."""
    root = resolve_notebook_root() / "docs"
    return sorted(path.name for path in root.glob("*.md"))

@tool
def read_course_doc(name: str) -> str:
    """Read a single markdown document from the docs folder."""
    path = resolve_notebook_root() / "docs" / name
    if not path.exists():
        return f"Document {name} does not exist."
    return path.read_text(encoding="utf-8")[:1000]

@tool
def summarize_learning_goal(step: int) -> str:
    """Return a short learning-goal summary for a core course step."""
    summaries = {
        4: "Understand tool definitions, signatures, and registration.",
        9: "Build a local RAG loop with retrieval and grounded prompting.",
        12: "Compare orchestration patterns and choose the right one.",
    }
    return summaries.get(step, "No summary registered for that step.")


## Part 2: Core Implementation

### Ask the model to create a tool plan


In [ ]:
planner = create_agent(
    name="ToolPlanner",
    instructions=(
        "You are a systems-thinking assistant. "
        "When you answer, explain which tool seems appropriate and why."
    ),
    tools=[list_course_docs, read_course_doc, summarize_learning_goal],
)

plan_response = await planner.run(
    "I need the docs file that best explains provider setup, and also the goal of Step 12."
)
print(plan_response.text)


## Part 2: Core Implementation

### Inspect a manual multi-step tool workflow


In [ ]:
docs = list_course_docs()
provider_doc_name = next((name for name in docs if "providers" in name), docs[0])
provider_doc_excerpt = read_course_doc(provider_doc_name)
step_goal = summarize_learning_goal(12)

print_json(
    {
        "available_docs": docs,
        "chosen_doc": provider_doc_name,
        "step_goal": step_goal,
        "excerpt_preview": provider_doc_excerpt[:240],
    },
    title="Manual tool chain inspection",
)


## Part 3: Hands-On Exercises

### Exercise 1
Refactor a weak tool so it returns a stable dictionary payload with clear field names and no ambiguous prose.

### Exercise 2
Design two tool descriptions so the model can clearly distinguish when to list documents versus when to read one document.

This solution notebook includes completed code for both guided exercises.


In [ ]:
def strong_lookup(topic: str) -> dict:
    return {
        "topic": topic,
        "status": "ok",
        "source": "local_reference",
        "summary": f"Helpful notes about {topic}",
    }

print_json(strong_lookup("workflow routing"), title="Exercise 1 solution")


In [ ]:
tool_a_description = "List document names only. Use this when the user does not yet know which file they need."
tool_b_description = "Read one named document. Use this only after the target file is already known."

print_json(
    {"list_tool": tool_a_description, "read_tool": tool_b_description},
    title="Exercise 2 solution",
)


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
def strong_lookup(topic: str) -> dict:
    return {
        "topic": topic,
        "status": "ok",
        "source": "local_reference",
        "summary": f"Helpful notes about {topic}",
    }

print_json(strong_lookup("workflow routing"), title="Exercise 1 solution")


In [ ]:
tool_a_description = "List document names only. Use this when the user does not yet know which file they need."
tool_b_description = "Read one named document. Use this only after the target file is already known."

print_json(
    {"list_tool": tool_a_description, "read_tool": tool_b_description},
    title="Exercise 2 solution",
)


## Part 5: Best Practices & Tips

        - Prefer structured outputs whenever another tool or prompt step will consume the result.
- Keep tool purpose boundaries sharp so the model can choose reliably.
- Treat tool descriptions as part of the interface contract, not optional decoration.
- Inspect manual tool chains before assuming the model should compose them automatically.


## Reflection & Next Experiments

- Which part of `Intermediate: Tool Reasoning and Schema Design` felt like the biggest jump from the core course?
- What would you keep deterministic before turning this into a live production feature?
- Where would you add tests, traces, or operator controls before shipping this pattern?


## Summary & Key Takeaways

You moved from basic tools to deliberate tool-interface design, with stronger attention to schema shape, overlap, and multi-step planning.


## Additional Resources

        - `helpers/llm.py`
- `docs/references.md`
- `core notebook: 04_understanding_tools.ipynb`
